# Temperature, Top-k & Top-p Sampling Comparison

This notebook compares the three core sampling strategies for autoregressive
text generation, both visually and in generated output.

$$
\begin{aligned}
\text{argmax:}\quad & x_{t+1} = \arg\max_v \, p_\theta(v \mid x_{1:t}) \\[4pt]
\text{Temperature:}\quad & p_\theta'(v) \propto \exp(\ell_v / \tau) \\[4pt]
\text{Top-}k\text{:}\quad & V_k = \{v \mid \ell_v \text{ in top }k\}, \quad
p_\theta'(v) = 0 \;\forall v \notin V_k \\[4pt]
\text{Top-}p\text{:}\quad & V_p = \min_{V} \sum_{v \in V} p_\theta(v) \geq p, \quad
p_\theta'(v) = 0 \;\forall v \notin V_p
\end{aligned}
$$

### What each strategy does

| Strategy | Parameter | Effect |
|---|---|---|
| **Temperature ($\tau$)** | $\tau \in (0, \infty)$ | Scales logits: $\tau \to 0$ = argmax, $\tau = 1$ = model distribution, $\tau \to \infty$ = uniform |
| **Top-$k$** | $k \in \mathbb{N}$ | Keeps only the $k$ most probable tokens, redistributes probability mass |
| **Top-$p$ (nucleus)** | $p \in (0, 1]$ | Keeps the smallest set of tokens whose cumulative probability exceeds $p$ |

These strategies are **compositional** — they are applied in order:
`temperature → top-k → top-p`. The `sample()` function in
`core.transformer.sampling` implements this pipeline.

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import math

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from core.transformer import GPT, BPETokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {DEVICE}")

---
## 1. Model & Sample Prompt

We need a model to produce logits. The notebook first tries to load a
checkpoint from `train_gpt.ipynb`; if none exists, it creates a small
untrained GPT. Even an untrained model produces logits suitable for
demonstrating the sampling mechanics.

In [ ]:
# ── Load trained model config, tokenizer, and weights from models/ ───
MODEL_DIR = Path(project_root) / "models"

if (MODEL_DIR / "training_config.pt").exists():
    config = torch.load(
        MODEL_DIR / "training_config.pt", map_location="cpu", weights_only=True
    )
    print("=== Training Config ===")
    for k, v in config.items():
        print(f"  {k}: {v}")

    # Load BPE tokenizer
    tokenizer_data = torch.load(
        MODEL_DIR / "bpe_tokenizer.pt", map_location="cpu", weights_only=True
    )
    tokenizer = BPETokenizer(
        vocab_size=len(tokenizer_data["vocab"]),
        special_tokens=tokenizer_data["special_tokens"],
        regex_pattern=tokenizer_data["regex_pattern"],
    )
    tokenizer.vocab = tokenizer_data["vocab"]
    tokenizer.id_to_token = tokenizer_data["id_to_token"]
    tokenizer.merges = tokenizer_data["merges"]
    tokenizer.merge_ranks = tokenizer_data["merge_ranks"]
    print(f"\nBPE tokenizer: vocab_size={tokenizer.vocab_size}")

    # Phase 6: pass through GQA (n_kv_heads) and MoE params if present
    model = GPT(
        vocab_size=config["vocab_size"],
        d_model=config["d_model"],
        n_layers=config["n_layers"],
        n_heads=config["n_heads"],
        n_kv_heads=config.get("n_kv_heads"),
        max_seq_len=config["block_size"],
        d_ff=config["d_ff"],
        dropout=0.0,
        use_moe=config.get("use_moe", False),
        n_experts=config.get("n_experts", 8),
        moe_k=config.get("moe_k", 2),
    )
    model.load_state_dict(
        torch.load(
            MODEL_DIR / "gpt_tinyshakespeare.pt",
            map_location="cpu",
            weights_only=True,
        )
    )
    print("Loaded trained checkpoint ✓")
    VOCAB_SIZE = config["vocab_size"]
else:
    print("No trained model found — using untrained model")
    print("(Sampling mechanics will still work, but output will be random)")
    tokenizer = BPETokenizer(vocab_size=512)
    # Quick-train on a small sample so tokenizer has a valid vocab
    tokenizer.train(["ROMEO: But, soft! what light through yonder window breaks?"])
    model = GPT(512, 128, 4, 4, 256)
    VOCAB_SIZE = 512

model.eval()
model.to(DEVICE)
print(f"Model: {model}")
print(f"VOCAB_SIZE = {VOCAB_SIZE}")

In [ ]:
prompt_texts = [
    "ROMEO: But, soft! what light through yonder window breaks?",
    "JULIET: O Romeo, Romeo! wherefore art thou Romeo?",
    "To be, or not to be, that is the question:",
]


def encode(prompt: str) -> torch.LongTensor:
    ids = tokenizer.encode(prompt)
    return torch.tensor([ids], dtype=torch.long, device=DEVICE)


def decode(ids: list[int]) -> str:
    return tokenizer.decode(ids, skip_special_tokens=True)


prompt = prompt_texts[0]
token_ids = encode(prompt)
print(f"Prompt: {prompt}")
print(
    f"Tokens ({token_ids.size(1)} tokens): {token_ids.tolist()[0][:15]}{'...' if token_ids.size(1) > 15 else ''}"
)

---
## 2. Extracting Raw Logits

Run a forward pass and capture the logits for the **last position**.
This is the distribution the model uses to predict the next token.

In [ ]:
def _safe_printchar(c: str) -> str:
    """Escape control chars for console display; leave printable chars alone."""
    if c == "\n":
        return r"\n"
    if c == "\t":
        return r"\t"
    if c == "\r":
        return r"\r"
    if c == " ":
        return r"<sp>"
    if c == "\x00":
        return r"\0"
    return c


with torch.no_grad():
    logits_full = model(token_ids)  # (1, seq_len, vocab_size)
    logits = logits_full[0, -1, :]  # (vocab_size,)  — last position

probs_raw = F.softmax(logits, dim=-1)

# Find the top tokens for display
top_probs, top_indices = probs_raw.topk(10)

print(f"Logits shape: {logits.shape}")
print(f"Max logit: {logits.max().item():.3f}, Min: {logits.min().item():.3f}")
print(
    f"Distribution entropy: "
    f"{-(probs_raw * torch.log(probs_raw.clamp(min=1e-10))).sum().item():.3f}"
)
print("\nTop 10 most likely tokens:")
for p, idx in zip(top_probs.cpu(), top_indices.cpu()):
    char = tokenizer.id_to_token.get(idx.item(), "?")
    print(f"  '{_safe_printchar(char)}' (idx {idx:2d}): {p.item():.4f}")

---
## 3. Effect of Temperature

Temperature $\tau$ scales the logits **before** softmax:

$$p_\tau(v) = \frac{\exp(\ell_v / \tau)}{\sum_{w} \exp(\ell_w / \tau)}$$

| $\tau$ | Behaviour |
|---|---|
| $\tau \to 0$ | Argmax — always pick the most likely token |
| $\tau = 1$ | Original model distribution |
| $\tau > 1$ | Distribution becomes more uniform ("flattened") |
| $\tau \to \infty$ | Uniform distribution — all tokens equally likely |

Dividing by $\tau \ll 1$ amplifies the differences between logits,
so softmax becomes "sharper" (more peaked).
Dividing by $\tau \gg 1$ shrinks differences toward zero, so softmax
approaches uniform.

In [ ]:
def _safe_chartok(c: str) -> str:
    """Replace invisible/whitespace chars with visible labels for chart axes."""
    REPLACE = {
        "\n": r"<newline>",
        "\t": r"<tab>",
        "\r": r"<CR>",
        " ": r"<space>",
        "\x00": r"<null>",
    }
    return REPLACE.get(c, c)


temperatures = [0.1, 0.5, 0.8, 1.0, 1.5, 3.0]
n_tokens = 20  # number of top tokens to display

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(
    "Effect of Temperature on the Next-Token Distribution", fontsize=15, y=1.02
)

for ax, tau in zip(axes.flat, temperatures):
    scaled = logits / max(tau, 1e-8)
    probs = F.softmax(scaled, dim=-1)
    top_p, top_i = probs.topk(n_tokens)
    chars = [
        _safe_chartok(tokenizer.id_to_token.get(i.item(), "?")) for i in top_i.cpu()
    ]

    colors = plt.cm.Blues(0.3 + 0.7 * top_p.cpu().numpy())
    ax.bar(
        range(n_tokens), top_p.cpu(), color=colors, edgecolor="steelblue", linewidth=0.5
    )
    ax.set_xticks(range(n_tokens))
    ax.set_xticklabels(chars, fontsize=8)
    ax.set_title(f"$\\tau = {tau}$", fontsize=13)
    ax.set_ylabel("Probability")
    ax.set_ylim(0, 1.05)

    # Annotate entropy
    H = -(probs * torch.log(probs.clamp(min=1e-10))).sum().item()
    H_max = math.log(VOCAB_SIZE)
    ax.text(
        0.95,
        0.95,
        f"H={H:.2f}\n(max={H_max:.1f})",
        transform=ax.transAxes,
        va="top",
        ha="right",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
    )

plt.tight_layout()
plt.show()

---
## 4. Effect of Top-$k$ Sampling

Top-$k$ keeps only the $k$ most probable tokens and sets the rest to
zero probability. The surviving mass is renormalised:

$$p'_k(v) = \begin{cases}
\dfrac{p(v)}{\sum_{w \in V_k} p(w)} & \text{if } v \in V_k \\[6pt]
0 & \text{otherwise}
\end{cases}$$

where $V_k = \{v \mid \ell_v \text{ is in top } k\}$.

This prevents sampling from the **long tail** of very unlikely tokens.
Small $k$ = very conservative (only a few options).
Large $k$ = more permissive (approaches the full distribution).

In [ ]:
def _safe_chartok(c: str) -> str:
    """Replace invisible/whitespace chars with visible labels for chart axes."""
    REPLACE = {
        "\n": r"<newline>",
        "\t": r"<tab>",
        "\r": r"<CR>",
        " ": r"<space>",
        "\x00": r"<null>",
    }
    return REPLACE.get(c, c)


topk_values = [1, 3, 10, 40]

fig, axes = plt.subplots(1, len(topk_values), figsize=(16, 4))
fig.suptitle("Effect of Top-k Filtering", fontsize=15, y=1.05)

for ax, k in zip(axes, topk_values):
    # Simulate top-k: mask out tokens below the k-th threshold
    vals, idxs = probs_raw.topk(k)
    threshold = vals[-1].item()
    masked = probs_raw.clone()
    masked[masked < threshold] = 0.0
    masked = masked / masked.sum()  # renormalise

    top_p, top_i = masked.topk(min(k + 5, VOCAB_SIZE))
    chars = [
        _safe_chartok(tokenizer.id_to_token.get(i.item(), "?")) for i in top_i.cpu()
    ]
    survived = (top_p > 0).sum().item()

    colors = ["steelblue" if p > 0 else "lightgray" for p in top_p.cpu()]
    ax.bar(
        range(len(top_p)),
        top_p.cpu(),
        color=colors,
        edgecolor="steelblue",
        linewidth=0.5,
    )
    ax.axvline(
        x=survived - 0.5,
        color="red",
        linestyle="--",
        alpha=0.6,
        label="top-k threshold",
    )

    ax.set_xticks(range(len(top_p)))
    ax.set_xticklabels(chars, fontsize=8, rotation=45)
    ax.set_title(f"top-$k$ = {k} ( {survived} tokens survive )", fontsize=12)
    ax.set_ylabel("Probability")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

---
## 5. Effect of Top-$p$ (Nucleus) Sampling

Top-$p$ (nucleus sampling) keeps the **smallest** set of tokens whose
cumulative probability exceeds $p$:

$$V_p = \min\{V \mid \sum_{v \in V} p(v) \geq p\}$$

Unlike top-$k$, which always keeps exactly $k$ tokens, top-$p$ adapts:
- When the distribution is **sharp** (one token dominates), it keeps very
  few tokens (e.g. $p=0.9$ might keep just 1-2 tokens)
- When the distribution is **flat** (many tokens plausible), it keeps more

This makes it dynamic — it cuts off the tail where cumulative probability
is negligible, without a hard count limit.

In [ ]:
def _safe_chartok(c: str) -> str:
    """Replace invisible/whitespace chars with visible labels for chart axes."""
    REPLACE = {
        "\n": r"<newline>",
        "\t": r"<tab>",
        "\r": r"<CR>",
        " ": r"<space>",
        "\x00": r"<null>",
    }
    return REPLACE.get(c, c)


topp_values = [0.5, 0.8, 0.9, 0.99]

fig, axes = plt.subplots(1, len(topp_values), figsize=(16, 4))
fig.suptitle("Effect of Top-p (Nucleus) Filtering", fontsize=15, y=1.05)

for ax, p in zip(axes, topp_values):
    sorted_probs, sorted_indices = probs_raw.sort(descending=True)
    cumsum = sorted_probs.cumsum(dim=-1)

    # Find cutoff: smallest set with cumulative prob > p
    mask = cumsum - sorted_probs > p
    n_survive = int((~mask).sum().item())

    # Build filtered distribution
    filtered = sorted_probs.clone()
    filtered[mask] = 0.0
    filtered = filtered / filtered.sum()  # renormalise

    # Show top tokens
    display_n = min(n_survive + 5, VOCAB_SIZE)
    top_p, top_i = filtered.topk(display_n)
    chars = [
        _safe_chartok(tokenizer.id_to_token.get(sorted_indices[i].item(), "?"))
        for i in range(display_n)
    ]

    survived = (top_p > 0).sum().item()
    colors = ["steelblue" if i < survived else "lightgray" for i in range(len(top_p))]

    ax.bar(
        range(display_n),
        top_p.cpu(),
        color=colors,
        edgecolor="steelblue",
        linewidth=0.5,
    )
    ax.axvline(
        x=survived - 0.5,
        color="red",
        linestyle="--",
        alpha=0.6,
        label=f"{survived} tokens survive",
    )

    ax.set_xticks(range(display_n))
    ax.set_xticklabels(chars, fontsize=8, rotation=45)
    ax.set_title(f"top-$p$ = {p} ( {survived} tokens )", fontsize=12)
    ax.set_ylabel("Probability")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

---
## 7. Side-by-Side Text Generation

Now let's generate actual text with each strategy and compare the output.
We'll use the same prompt but sample differently.

### Strategries tested

| Label | Params | Expected behaviour |
|---|---|---|
| **Argmax** | $\tau=0$ | Deterministic, repetitive, rigid |
| **Low temp** | $\tau=0.5$ | Conservative, high-probability tokens only |
| **Default** | $\tau=0.8,\, k=40$ | Balanced creativity / coherence |
| **Nucleus** | $\tau=0.8,\, p=0.9$ | Dynamic filtering, popular default |
| **High temp** | $\tau=1.2,\, p=0.9$ | Creative but may drift |
| **Very high** | $\tau=2.0$ | Almost random, often incoherent |

> **Note**: If using an untrained model, the output will be garbage
> (random characters). The comparison is still valid — the point is
> seeing how *behaviour* changes across strategies, not output quality.

In [ ]:
strategies = [
    ("Argmax", dict(temperature=0.0)),
    ("Conservative", dict(temperature=0.5, top_k=10)),
    ("Balanced", dict(temperature=0.8, top_k=40)),
    ("Nucleus (p=0.9)", dict(temperature=0.8, top_p=0.9)),
    ("Creative", dict(temperature=1.2, top_p=0.9)),
    ("High entropy", dict(temperature=2.0)),
]

MAX_NEW = 200

results = {}
for label, kwargs in strategies:
    output_ids = model.generate(
        token_ids, max_new_tokens=MAX_NEW, eos_token_id=tokenizer.eos_id, **kwargs
    )
    text = decode(output_ids[0].tolist())
    results[label] = text
    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print(f"  Params: {kwargs}")
    print(f"  Generated {len(output_ids[0]) - token_ids.size(1)} tokens")
    print(f"{'=' * 60}")
    print(text[:500])

---
## 8. Quantitative Comparison

Beyond subjective quality, we can measure **diversity** and **entropy**
of the generated text:

- **Token entropy**: $H[P] = -\sum_v p(v) \log p(v)$ — how "surprised" the model is
- **Type-token ratio (TTR)**: unique tokens / total tokens — lexical diversity
- **Self-repetition**: fraction of repeated n-grams — measures stuck loops

In [ ]:
def token_entropy(text: str) -> float:
    """Empirical entropy of the token distribution."""
    ids = tokenizer.encode(text)
    counts = {}
    for i in ids:
        counts[i] = counts.get(i, 0) + 1
    total = len(ids)
    H = 0.0
    for c in counts.values():
        p = c / total
        H -= p * math.log2(p)
    return H


def type_token_ratio(text: str) -> float:
    """Unique tokens / total tokens."""
    ids = tokenizer.encode(text)
    return len(set(ids)) / max(1, len(ids))


def repetition_rate(text: str, ngram: int = 4) -> float:
    """Fraction of n-grams that are repeated."""
    ids = tokenizer.encode(text)
    if len(ids) < ngram:
        return 0.0
    seen = set()
    repeats = 0
    total = 0
    for i in range(len(ids) - ngram + 1):
        ng = tuple(ids[i : i + ngram])
        if ng in seen:
            repeats += 1
        seen.add(ng)
        total += 1
    return repeats / max(1, total)


print(f"{'Strategy':<20} {'Entropy':>8} {'TTR':>8} {'Rep(4-gram)':>12} {'Len':>6}")
print("-" * 56)

for label in [s[0] for s in strategies]:
    text = results[label]
    generated = text[len(prompt) :][:200]
    if not generated:
        continue
    H = token_entropy(generated)
    ttr = type_token_ratio(generated)
    rep = repetition_rate(generated)
    print(f"{label:<20} {H:>8.3f} {ttr:>8.3f} {rep:>12.3f} {len(generated):>6}")

---
## Summary: When to Use Each Strategy

| Strategy | Use when | Example |
|---|---|---|
| **Argmax** ($\tau=0$) | You need a deterministic, predictable completion | Autocomplete, code generation |
| **Low temp** ($\tau \leq 0.5$) | You want safe, high-probability text | Translation, factual QA |
| **Balanced** ($\tau \approx 0.8, k \approx 40$) | Good default — some creativity, low risk | Chat, story generation |
| **Nucleus** ($\tau=0.8, p=0.9$) | Industry default — adapts to distribution | GPT models, Llama, etc. |
| **High temp** ($\tau \approx 1.2$) | You want creative, surprising output | Brainstorming, poetry |
| **Very high** ($\tau > 1.5$) | You want degenerate / random text | Testing, art projects |

### The pipeline order matters

The `sample()` function applies strategies **in sequence**:

    raw logits → [temperature scaling] → [top-k mask] → [top-p mask]
               → [softmax] → [multinomial sample]

This means:
- Temperature first "softens" the logits
- Top-k then removes improbable tokens from the long tail
- Top-p then further filters based on cumulative probability
- Softmax normalises the surviving scores into a distribution
- Multinomial draws one token from the final distribution

The ordering is important: applying top-k *before* temperature makes
a difference, because temperature scaling changes which tokens are in
the top-k set.